# Leaflet cluster map of talk locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter getorg --upgrade
```

After which you can run this from the `_talks/` directory, via:

```bash
 jupyter nbconvert --to notebook --execute talkmap.ipynb --output talkmap_out.ipynb
```
 
The `_talks/` directory contains `.md` files of all your talks. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and uses the `getorg` library to output data, HTML, and Javascript for a standalone cluster map.

In [56]:
# Start by installing the dependencies
!pip install python-frontmatter getorg --upgrade
import frontmatter
import glob
import getorg
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut

In [57]:
# Collect the Markdown files
g = glob.glob("_talks/*.md")

In [58]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Prepare to geolocate
geocoder = Nominatim(user_agent="wings-uhm.github.io/Web/")
location_dict = {}
location = ""
permalink = ""
title = ""

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [59]:
# Perform geolocation
for file in g:
    # Read the file
    data = frontmatter.load(file)
    data = data.to_dict()

    # Press on if the location is not present
    if 'location' not in data:
        continue

    # Prepare the description
    title = data['title'].strip()
    venue = data['venue'].strip()
    location = data['location'].strip()
    date = data['date'].strftime("%b %d, %Y")
    url = data.get('url_link')
    permalink = data.get('permalink', '')
    ref_link = url if url else f"/Web{permalink}"
    description = f'<strong><a href="{ref_link}" target="_blank">{title}</a></strong> - {date}<br />{venue}, {location}'

    # Geocode the location and report the status
    try:
        location_dict[description] = geocoder.geocode(location, timeout=TIMEOUT)
        print(description, location_dict[description])
    except ValueError as ex:
        print(f"Error: geocode failed on input {location} with message {ex}")
    except GeocoderTimedOut as ex:
        print(f"Error: geocode timed out on input {location} with message {ex}")
    except Exception as ex:
        print(f"An unhandled exception occurred while processing input {location} with message {ex}")

<strong><a href="/Web/papers/2026_ICNC_Waveform.pdf" target="_blank">Waveform Design for ISAC: Trends and Future Directions</a></strong> - Feb 18, 2026<br />IEEE ICNC - 2026, Maui, HI, USA Maui, Maui County, Hawaii, United States
<strong><a href="https://ece.hawaii.edu/events/catalog/ECE-Seminars?evt=664" target="_blank">Intelligent Open RAN: AI-Driven Integrated Sensing and Communication (ISAC) for Trusted Next Generation Wireless Networks</a></strong> - Apr 17, 2025<br />College of Engineering, University of Hawaiʻi at Mānoa, Holmes Hall, 2540 Dole St, Honolulu, HI 96822 Holmes Hall, 2540, Dole Street, Moiliili, McCully, East Honolulu, Honolulu, Honolulu County, Hawaii, 96822, United States
<strong><a href="https://luna-xue.github.io/assets/files/Enhancing%20Security%20and%20Privacy%20in%20Distributed%20Wireless%20Networks%20Through%20Physical%20Layer%20Techniques%20_%20Stevens%20Institute%20of%20Technology.html" target="_blank">Enhancing Security and Privacy in Distributed Wireless 

In [60]:
# Save the map
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(location_dict, folder_name="talkmap", hashed_usernames=False)

'Written map to talkmap/'

In [61]:
from pathlib import Path
import re

map_file = Path("talkmap/map.html")
content = map_file.read_text()


# Customize the map's initial view and zoom level
content = re.sub(
    r"latlng\s*=\s*L\.latLng\([^)]*\);",
    "latlng = L.latLng(30, -110);",
    content
)

content = re.sub(
    r"zoom\s*:\s*[\d.]+",
    "zoom: 3",
    content
)

content = re.sub(
    r"map\.zoomIn\(\);?",
    "",
    content
)

map_file.write_text(content)

1834

## Debug for future customization
add this when customizing map view to talkmap/map.html to console trace and set values for zoom and lat/long in above block
```js
map.on('click', function () {
                var c = map.getCenter();
                console.log('Center:', c.lat, c.lng, 'Zoom:', map.getZoom());
            });
```